In [1]:
import pocketeer as pt

# Load structure from PDB file
atomarray = pt.load_structure("6WAU.pdb")

# Detect pockets with default parameters
pockets = pt.find_pockets(atomarray)

# Print results
for pocket in pockets:
    print(f"Pocket {pocket.pocket_id}:")
    print(f"  Score: {pocket.score:.2f}")
    print(f"  Volume: {pocket.volume:.1f} Å³")
    print(f"  Spheres: {pocket.n_spheres}")

Pocket 0:
  Score: 5.12
  Volume: 1181.2 Å³
  Spheres: 38
Pocket 1:
  Score: 4.79
  Volume: 684.1 Å³
  Spheres: 71


In [2]:
pocket_score_threshold = 5
pockets = [p for p in pockets if p.score > pocket_score_threshold]

In [3]:
pt.view_pockets(atomarray, pockets)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Chemistry-colored alpha-spheres and ROCS-like ligand overlay

This section keeps ligand preparation separate from pocket coloring.

1. `ligand_preparation.py` converts SMILES or ligand files into 3D conformers and color/pharmacophore points.
2. `pocket_coloring.py` colors `pocketeer` alpha-spheres by local protein chemistry and scores ligand overlays with ROCS-like `ShapeTanimoto`, `ColorTanimoto`, and `TanimotoCombo`.

Run the original `pocketeer` cells first so `atomarray` and `pockets` already exist.

In [4]:
import py3Dmol

In [5]:
from ligand_preparation import prepare_ligand_from_smiles
from pocket_coloring import color_pockets
from overlay import rank_ligand_over_pockets
from visualization import view_pocket_overlay, summarize_overlay

ligand_smiles = "BrC1=NN(C)C2=C1C(N3CC[C@@H](NC4CCC4)CCC3)=NC(N5CCC(F)(F)C5)=N2"
prepared_ligand = prepare_ligand_from_smiles(
    ligand_smiles,
    name="UNC10415778",
    num_conformers=10,
)

'''
ligand_smiles = "CN(CC1(CC1)c1ccc(F)cc1)C(=O)[C@H](Cc1csc2ccccc12)Nc1cc(ncn1)C(N)=O"
prepared_ligand = prepare_ligand_from_smiles(
    ligand_smiles,
    name="A46",
    num_conformers=10,
)

ligand_smiles = "c1nc(c2c(n1)n(cn2)[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O)N"
prepared_ligand = prepare_ligand_from_smiles(
    ligand_smiles,
    name="ATP",
    num_conformers=10,
)
'''

colored_pockets = color_pockets(atomarray, pockets)
overlay_results = rank_ligand_over_pockets(prepared_ligand, colored_pockets)

for result in overlay_results:
    summary = summarize_overlay(result)
    pocket_id = summary['pocket_id']
    pocket = next(p for p in pockets if p.pocket_id == pocket_id)
    pocket_score = getattr(pocket, 'score', None)
    print(
        f"Pocket {summary['pocket_id']}: "
        f"Pocket Score={pocket_score}  "
        f"Score={summary['gaussian_fit_score']:+.2f}  "
    )

Pocket 0: Pocket Score=5.1225  Score=+5.64  


In [7]:
import py3Dmol
import visualization

# Manually inject py3Dmol into the draco visualization module
visualization.py3Dmol = py3Dmol

# Now try running your code again
best_result = overlay_results[0]
best_pocket = next(
    pocket for pocket in colored_pockets if pocket.pocket_id == best_result.pocket_id
)

viewer = view_pocket_overlay(
    atomarray,
    best_pocket,
    best_result,
    receptor_cartoon=True,
    receptor_surface=False,
)
viewer

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
import py3Dmol

with open('test_output/pose_016_pocket_012_rank_+10_conf_3.pdb', 'r') as pdb_file:
    pdb_data = pdb_file.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_data, 'pdb')

# Set cartoon style for protein (default A chain)
view.setStyle({'model': -1, 'cartoon': {'color': 'spectrum'}})

# Try to color showing ligands (if present): show as sticks
# We'll guess the ligand is any residue that's not standard AA or nucleic acid; in PDBs, hetatm records are usually used, but OpenMM "ATOM" is used for everything sometimes.
# Let's highlight non-protein atoms as sticks (heuristic: residue number > 200 or not in list of AA)
protein_resnames = [
    'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLU', 'GLN', 'GLY', 'HIS', 'ILE',
    'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL'
]

# Use py3Dmol selection strings: select all nonstandard residues
view.addStyle({'and': [
    {'not': {'resi': list(range(1, 1000))}},  # unlikely, so they'll be ignored
    # fudge: select only non-protein
    {'not': {'resn': protein_resnames}}
]}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.3}})

# Fallback: highlight all non-protein residues as sticks
view.addStyle({'not': {'resn': protein_resnames}}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.3}})

view.zoomTo()
view.show()

view

FileNotFoundError: [Errno 2] No such file or directory: 'test_output/pose_016_pocket_012_rank_+10_conf_3.pdb'